### このノートブックの目的

このノートブックでは，Graphデータを用いた機械学習を行うためのモジュール

- PyTorch Geometric (PyG)
- NetworkX

の使い方について学ぶことを目的とする．

In [1]:
import networkx as nx
import matplotlib.pyplot as plt
import duckdb as db
import pandas as pd
import numpy as np
from pathlib import Path
from string import Template

まず，ある日付の株価データを取得する．<br>
そして，取得したプライスデータの相関に基づいたグラフが作成可能かどうかを検証する．

In [6]:
PATH_TO_DATABASE = "../db/duckdb/equities_bars_daily.duckdb"
TABLE_NAME = "eqt_main_tmp"

In [3]:
class Templates:

    @property
    def isedge(self):
        with open(Path("sql/get_corr.sql"), "r", encoding="utf-8") as f:
            """
            window_size: int, threshold: float, TradeDate: str
            """
            return Template(f.read())

    @property
    def get_Codes_by_TradeDate(self):
        return Template(
            f"""
            SELECT DISTINCT TradeDate, Code
            FROM {TABLE_NAME}
            WHERE TradeDate = '$TradeDate'
            ORDER BY TradeDate, Code
            """
        )

    @property
    def get_TradeDates(self):
        return Template(
            f"""
            SELECT DISTINCT TradeDate
            FROM {TABLE_NAME}
            ORDER BY TradeDate
            """
        )

    

In [4]:
def send_query(query: str):
    print('sending query...')
    with db.connect(PATH_TO_DATABASE) as con:
        df = con.sql(query).df()
    print('get result...')
    return df

In [ ]:
def graphize(result: pd.DataFrame, TradeDate: str, graph: nx.Graph):
    max_count = len(result)
    test_count = 0
    for _, row in result.iterrows():
        test_count += 1
        if test_count % 10000 == 0 or test_count == max_count:
            print(f"\r ({test_count*100/max_count:.2%}/{100}) Adding edge: {row['t1_Code']} - {row['t2_Code']} (TradeDate: {TradeDate})", end=" ")
        a_code = row["t1_Code"]
        b_code = row["t2_Code"]
        a_code = f"{a_code}\n({TradeDate})"
        b_code = f"{b_code}\n({TradeDate})"
        graph.add_edge(a_code, b_code)
    print("\n")
    return graph

def send_and_get():
    TMPs = Templates()
    graph = nx.Graph()

    TradeDateList = send_query(TMPs.get_TradeDates.substitute())
    # print(f"TradeDateList: {TradeDateList}")
    for TradeDate in TradeDateList["TradeDate"].to_list(): # 99:100 はデバッグ用に1日だけ処理するためのスライス
        TradeDate = TradeDate.strftime('%Y-%m-%d')
        print(f"Processing TradeDate: {TradeDate}")
        # また，コードの組み合わせは順序を考慮しないため，i <= j の組み合わせのみを考慮する．
        query = TMPs.isedge.substitute(
            window_size=75,
            threshold=0.8,
            TradeDate=TradeDate
        )
        result = send_query(query)
        graph = graphize(result, TradeDate, graph)

In [8]:
TMPs = Templates()
TradeDateList = send_query(TMPs.get_TradeDates.substitute())
print(f"TradeDateList: {TradeDateList}")

sending query...
get result...
TradeDateList:       TradeDate
0    2008-05-07
1    2008-05-08
2    2008-05-09
3    2008-05-12
4    2008-05-13
...         ...
4387 2026-04-13
4388 2026-04-14
4389 2026-04-15
4390 2026-04-16
4391 2026-04-17

[4392 rows x 1 columns]
